# TabM Baseline for Playground Series S6E9

This notebook is designed for Google Colab GPU. It follows the high-value idea from Yusuke Hayashi's TabM notebook: use a tabular neural network with raw features, digit-token features, and exact income frequency.

Expected outputs:

- `submission_tabm_colab.csv`
- `oof_tabm_colab.csv`
- optional blended submissions if you upload one of our existing tree-model submissions

Use GPU runtime in Colab before running: `Runtime -> Change runtime type -> GPU`.

## 1. Install TabM Dependencies

This keeps Colab's preinstalled PyTorch build and installs `pytabkit` against it. Restarting the runtime should not be necessary, but if Colab asks for it, restart and continue from the next cell.

In [1]:
import subprocess
import sys

import torch

with open("constraints.txt", "w", encoding="utf-8") as f:
    f.write(f"torch=={torch.__version__}\n")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-c", "constraints.txt", "pytabkit==1.7.3"],
    check=True,
)

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch 2.11.0+cu128
cuda available: True
gpu: NVIDIA A100-SXM4-80GB


## 2. Imports and Settings

`FAST=True` is the recommended first run. It should be much faster and still strong. Later, try `FAST=False` for a slower but potentially better run.

In [2]:
import glob
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from pytabkit import TabM_D_Classifier
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore", category=FutureWarning, message=".*force_all_finite.*")

SMOKE = False
FAST = True
SEED = 42
N_SPLITS = 5
TARGET = "Will_Buy_EV"
ID = "id"

def pick_device() -> str:
    if not torch.cuda.is_available():
        return "cpu"
    try:
        (torch.ones(8, device="cuda") @ torch.ones(8, device="cuda")).item()
        return "cuda"
    except Exception as error:
        print("CUDA unusable, falling back to CPU:", error)
        return "cpu"

DEVICE = pick_device()
print("device:", DEVICE)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

device: cuda
gpu: NVIDIA A100-SXM4-80GB


## 3. Load Data

Put `train.csv`, `test.csv`, and `sample_submission.csv` somewhere under `/content`, or mount Google Drive and put them there. This cell searches recursively.

In [8]:
from google.colab import drive
drive.mount('/content/drive')

def find_file(name: str) -> str:

    search_roots = ["/content", "/content/drive", "."]
    for root in search_roots:
        hits = glob.glob(f"{root}/**/{name}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"Could not find {name}. Upload it or mount Drive first.")

TRAIN_PATH = find_file("train.csv")
TEST_PATH = find_file("test.csv")
SAMPLE_PATH = find_file("sample_submission.csv")

print("train:", TRAIN_PATH)
print("test:", TEST_PATH)
print("sample:", SAMPLE_PATH)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

y = train[TARGET].map({"No": 0, "Yes": 1}).astype(int).to_numpy()
print("train shape:", train.shape)
print("test shape:", test.shape)
print("positive rate:", round(float(y.mean()), 6))
assert sample.columns.tolist() == [ID, TARGET]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train: /content/drive/MyDrive/playground-series-s6e9/train.csv
test: /content/drive/MyDrive/playground-series-s6e9/test.csv
sample: /content/drive/MyDrive/playground-series-s6e9/sample_submission.csv
train shape: (668665, 15)
test shape: (286571, 14)
positive rate: 0.174645


## 4. Feature Engineering

This follows Yusuke's TabM recipe:

- raw numeric features
- categorical string features for embeddings
- yes/no flags as numeric 0/1
- exact `Annual_Income_USD` frequency over train + test
- digit tokens from numeric columns

All features are target-free, so train + test frequency/digit construction is safe.

In [11]:
NUMERIC = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]
CATEGORICAL = ["Gender", "City_Type", "Current_Car_Type", "Range_Anxiety_Level"]
FLAGS = ["Home_Charging_Possible", "Subsidy_Available"]
DIGIT_SPECS = {
    "Age": (1.0, 2),
    "Annual_Income_USD": (1.0, 6),
    "Daily_Commute_km": (10.0, 4),
    "Number_of_Cars_Owned": (1.0, 1),
    "Charging_Stations_Near_Home": (1.0, 2),
    "Charging_Stations_Near_Work": (1.0, 2),
    "Environmental_Concern_Level": (1.0, 1),
}

def build_features(frame: pd.DataFrame, income_counts: pd.Series) -> pd.DataFrame:
    out = pd.DataFrame(index=frame.index)

    for column in NUMERIC:
        out[column] = pd.to_numeric(frame[column], errors="coerce").astype(float)

    for column in CATEGORICAL:
        out[column] = frame[column].fillna("__MISSING__").astype(str)

    for column in FLAGS:
        out[column] = (
            frame[column]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({"yes": 1, "true": 1, "1": 1, "no": 0, "false": 0, "0": 0})
            .astype(float)
        )

    out["income_freq"] = frame["Annual_Income_USD"].map(income_counts).fillna(0).astype(float)

    for column, (scale, width) in DIGIT_SPECS.items():
        scaled = (pd.to_numeric(frame[column], errors="coerce") * scale).round().astype("Int64")
        for position in range(width):
            digit = ((scaled.abs() // 10**position) % 10).astype("Int64")
            out[f"digit__{column}__{position}"] = digit.astype("string").fillna("__MISSING__").astype(str)

    return out

income_counts = pd.concat([train["Annual_Income_USD"], test["Annual_Income_USD"]]).value_counts()
X = build_features(train, income_counts)
X_test = build_features(test, income_counts)
CAT_COLS = [column for column in X.columns if not pd.api.types.is_numeric_dtype(X[column])]

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)
print("categorical columns:", len(CAT_COLS))
print(CAT_COLS)

X shape: (668665, 32)
X_test shape: (286571, 32)
categorical columns: 22
['Gender', 'City_Type', 'Current_Car_Type', 'Range_Anxiety_Level', 'digit__Age__0', 'digit__Age__1', 'digit__Annual_Income_USD__0', 'digit__Annual_Income_USD__1', 'digit__Annual_Income_USD__2', 'digit__Annual_Income_USD__3', 'digit__Annual_Income_USD__4', 'digit__Annual_Income_USD__5', 'digit__Daily_Commute_km__0', 'digit__Daily_Commute_km__1', 'digit__Daily_Commute_km__2', 'digit__Daily_Commute_km__3', 'digit__Number_of_Cars_Owned__0', 'digit__Charging_Stations_Near_Home__0', 'digit__Charging_Stations_Near_Home__1', 'digit__Charging_Stations_Near_Work__0', 'digit__Charging_Stations_Near_Work__1', 'digit__Environmental_Concern_Level__0']


## 5. Train 5-Fold TabM

Fast mode uses smaller internal ensemble settings. For a stronger run, set `FAST=False` above and rerun from the settings cell.

In [12]:
if SMOKE:
    rng = np.random.default_rng(SEED)
    keep = rng.choice(len(X), 20_000, replace=False)
    X_train_model = X.iloc[keep].reset_index(drop=True)
    y_train_model = y[keep]
    X_test_model = X_test.iloc[:2_000].reset_index(drop=True)
    test_ids = test[ID].iloc[:2_000].reset_index(drop=True)
else:
    X_train_model = X.reset_index(drop=True)
    y_train_model = y
    X_test_model = X_test.reset_index(drop=True)
    test_ids = test[ID].reset_index(drop=True)

oof = np.zeros(len(X_train_model), dtype=float)
test_pred = np.zeros(len(X_test_model), dtype=float)
fold_assignments = np.full(len(X_train_model), -1, dtype=np.int8)
fold_scores = []
fold_times = []

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (fit_idx, val_idx) in enumerate(skf.split(X_train_model, y_train_model), start=1):
    start = time.time()
    model = TabM_D_Classifier(
        device=DEVICE,
        random_state=SEED + fold,
        n_threads=4,
        verbosity=0,
        val_metric_name="1-auc_ovr",
        tabm_k=8 if FAST else 32,
        batch_size=2048 if FAST else 256,
        lr=5e-3 if FAST else 2e-3,
        n_epochs=2 if SMOKE else (30 if FAST else 200),
        patience=5 if FAST else 16,
    )
    model.fit(
        X_train_model.iloc[fit_idx],
        y_train_model[fit_idx],
        X_val=X_train_model.iloc[val_idx],
        y_val=y_train_model[val_idx],
        cat_col_names=CAT_COLS,
    )

    oof[val_idx] = model.predict_proba(X_train_model.iloc[val_idx])[:, 1]
    test_pred += model.predict_proba(X_test_model)[:, 1] / N_SPLITS
    fold_assignments[val_idx] = fold

    fold_auc = roc_auc_score(y_train_model[val_idx], oof[val_idx])
    seconds = time.time() - start
    fold_scores.append(fold_auc)
    fold_times.append(seconds)
    print(f"fold {fold}: AUC={fold_auc:.6f}, seconds={seconds:.0f}")

overall_auc = roc_auc_score(y_train_model, oof)
print("OOF AUC:", round(overall_auc, 6))
print("fold mean:", round(float(np.mean(fold_scores)), 6))
print("fold std:", round(float(np.std(fold_scores)), 6))
print("total minutes:", round(float(np.sum(fold_times) / 60), 2))
assert (fold_assignments > 0).all()
assert np.isfinite(oof).all()
assert np.isfinite(test_pred).all()

fold 1: AUC=0.942961, seconds=50
fold 2: AUC=0.943763, seconds=47
fold 3: AUC=0.945015, seconds=52
fold 4: AUC=0.943992, seconds=45
fold 5: AUC=0.944012, seconds=48
OOF AUC: 0.943884
fold mean: 0.943948
fold std: 0.000656
total minutes: 4.03


## 6. Save TabM Outputs

Download these files from the Colab file panel, or copy them to Drive.

In [13]:
submission = pd.DataFrame({ID: test_ids.to_numpy(), TARGET: test_pred})
oof_frame = pd.DataFrame(
    {
        ID: train[ID].iloc[: len(oof)].to_numpy() if SMOKE else train[ID].to_numpy(),
        "target": y_train_model,
        "fold": fold_assignments,
        "tabm_pred": oof,
    }
)

submission.to_csv("submission_tabm_colab.csv", index=False)
oof_frame.to_csv("oof_tabm_colab.csv", index=False)

print("Saved submission_tabm_colab.csv")
print("Saved oof_tabm_colab.csv")
print(submission.shape)
print(submission.head())
print("valid probabilities:", submission[TARGET].between(0, 1).all())

Saved submission_tabm_colab.csv
Saved oof_tabm_colab.csv
(286571, 2)
       id  Will_Buy_EV
0  668665     0.016729
1  668666     0.012886
2  668667     0.005337
3  668668     0.003176
4  668669     0.023278
valid probabilities: True


## 7. Optional: Blend with Our Current Best Tree Submission

If you upload `submission_chris_lgbm_tuned_blend.csv` or `submission_chris_plus_lgbm_blend.csv` into Colab, this cell writes a few simple blends. We cannot compute an honest OOF blend unless we also upload the matching tree OOF predictions, so treat these as public-LB probes.

In [14]:
def minmax(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    lo = values.min()
    hi = values.max()
    if hi == lo:
        return np.zeros_like(values)
    return (values - lo) / (hi - lo)

def rank01(values: np.ndarray) -> np.ndarray:
    return minmax(rankdata(values, method="average"))

tree_candidates = glob.glob("/content/**/submission_chris_lgbm_tuned_blend.csv", recursive=True)
tree_candidates += glob.glob("/content/**/submission_chris_plus_lgbm_blend.csv", recursive=True)
tree_candidates += glob.glob("./**/submission_chris_lgbm_tuned_blend.csv", recursive=True)
tree_candidates += glob.glob("./**/submission_chris_plus_lgbm_blend.csv", recursive=True)

if not tree_candidates:
    print("No current-best tree submission found. Upload it to create blend files.")
else:
    tree_path = tree_candidates[0]
    tree_sub = pd.read_csv(tree_path)
    print("Using tree submission:", tree_path)
    assert tree_sub[ID].reset_index(drop=True).equals(submission[ID].reset_index(drop=True))

    tree_pred = tree_sub[TARGET].to_numpy(dtype=float)
    tabm_pred = submission[TARGET].to_numpy(dtype=float)

    blend_specs = {
        "submission_blend_50_tree_50_tabm.csv": 0.50 * tree_pred + 0.50 * tabm_pred,
        "submission_blend_60_tree_40_tabm.csv": 0.60 * tree_pred + 0.40 * tabm_pred,
        "submission_blend_70_tree_30_tabm.csv": 0.70 * tree_pred + 0.30 * tabm_pred,
        "submission_blend_rank_tree_tabm.csv": (rank01(tree_pred) + rank01(tabm_pred)) / 2,
    }

    for filename, pred in blend_specs.items():
        out = pd.DataFrame({ID: submission[ID], TARGET: pred})
        out.to_csv(filename, index=False)
        print(filename, out[TARGET].min(), out[TARGET].mean(), out[TARGET].max(), out[TARGET].between(0, 1).all())

Using tree submission: /content/drive/MyDrive/playground-series-s6e9/submission_chris_lgbm_tuned_blend.csv
submission_blend_50_tree_50_tabm.csv 2.982382563132546e-07 0.3372683738153971 0.9989339509356487 True
submission_blend_60_tree_40_tabm.csv 2.385906050506037e-07 0.36981435009621993 0.998942448763702 True
submission_blend_70_tree_30_tabm.csv 1.7894295378795277e-07 0.40236032637704255 0.9991545277190467 True
submission_blend_rank_tree_tabm.csv 0.0002355445440904491 0.5 0.999965976899187 True


In [20]:
out = pd.DataFrame({
    ID: submission[ID],
    TARGET: 0.30 * tree_pred + 0.70 * tabm_pred
})
out.to_csv("submission_blend_30_tree_70_tabm.csv", index=False)
print(out[TARGET].min(), out[TARGET].mean(), out[TARGET].max(), out[TARGET].between(0, 1).all())

4.175335588385564e-07 0.2721764212537517 0.9989304661004822 True


In [21]:
import os, glob

print("cwd:", os.getcwd())
print(glob.glob("submission_blend_*.csv"))
print(glob.glob("/content/submission_blend_*.csv"))

cwd: /content
['submission_blend_50_tree_50_tabm.csv', 'submission_blend_70_tree_30_tabm.csv', 'submission_blend_rank_tree_tabm.csv', 'submission_blend_60_tree_40_tabm.csv', 'submission_blend_30_tree_70_tabm.csv']
['/content/submission_blend_50_tree_50_tabm.csv', '/content/submission_blend_70_tree_30_tabm.csv', '/content/submission_blend_rank_tree_tabm.csv', '/content/submission_blend_60_tree_40_tabm.csv', '/content/submission_blend_30_tree_70_tabm.csv']


In [22]:
import shutil, glob, os

drive_dir = "/content/drive/MyDrive/playground-series-s6e9"
os.makedirs(drive_dir, exist_ok=True)

for path in glob.glob("submission_blend_*.csv"):
    dest = os.path.join(drive_dir, os.path.basename(path))
    shutil.copy(path, dest)
    print(dest)

/content/drive/MyDrive/playground-series-s6e9/submission_blend_50_tree_50_tabm.csv
/content/drive/MyDrive/playground-series-s6e9/submission_blend_70_tree_30_tabm.csv
/content/drive/MyDrive/playground-series-s6e9/submission_blend_rank_tree_tabm.csv
/content/drive/MyDrive/playground-series-s6e9/submission_blend_60_tree_40_tabm.csv
/content/drive/MyDrive/playground-series-s6e9/submission_blend_30_tree_70_tabm.csv
